# v6b MIXED baseline (GPU 0): all three languages at once, never switching
A new model from scratch, trained on Hindi, Chinese and Arabic **at the same time**: inside every phase the batches alternate Hindi, Chinese, Arabic, Hindi, ... (each batch is one language, so the script-restricted loss is unchanged). Same model, schedule and evaluation as the continual and Hindi-only runs.
Question: does a model that knows all three languages, but never switches, form the same kind of self?
**Just Run All.** Output appears below. Rerunning resumes from the last saved phase. Checkpoints: `runs/v6b_mixed_s0/ckpt/pXXX_mix.pt`.

In [1]:
import os, sys, glob, json, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # run this cell FIRST in a fresh kernel
os.environ["HF_HUB_DISABLE_XET"] = "1"          # HF uploads failed overnight in the new "xet" path; use the classic upload
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from common_v6 import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
RUN = "v6b_mixed_s0"

device: cuda | NVIDIA GeForce RTX 4090


In [2]:
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):                          # retry: HF sometimes drops the connection
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)

# FLORES dev + devtest
from datasets import load_dataset
for l, code in {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}.items():
    for split in ["dev", "devtest"]:
        p = f"data/flores/{split}.{l}.jsonl"
        if not os.path.exists(p):
            os.makedirs("data/flores", exist_ok=True)
            ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
            with open(p, "w", encoding="utf-8") as f:
                for t in ds["text"]:
                    f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
missing = [p for p in [f"data/en-{l}/{s}.jsonl" for l in LANGS for s in ["train", "val"]] if not os.path.exists(p)]
print("data OK" if not missing else f"MISSING: {missing}")

data OK


In [3]:
C = RunConfig(run_name=RUN, mode="mixed", n_phases=60, seed=0, time_cap_h=120.0, hf_repo=HF_REPO)
print(C)

RunConfig(run_name='v6b_mixed_s0', mode='mixed', order=('hi', 'zh', 'ar'), init_ckpt=None, n_phases=60, seed=0, max_tokens=16000, lr=0.0002, min_lr=2e-05, warmup_steps=500, cooldown_steps=2000, min_steps=2000, max_stable_steps=60000, eval_every=500, plateau_window=4, plateau_tol=0.005, weight_decay=0.1, grad_clip=1.0, masked=True, diverge_factor=1.15, max_retries=2, time_cap_h=120.0, hf_repo='s8n29/cml-lang-self-checkpoints', start_phase=None, noise_per_step=0.0, boundary_push=0.0, health_every=1, redo_every=0, redo_thresh=0.01)


In [4]:
D = load_data(train_langs=["hi", "zh", "ar"])

  hi: 993,016 train / 2,000 val
  zh: 755,626 train / 2,000 val
  ar: 998,000 train / 2,000 val
data loaded in 47s


In [ ]:
model = train_run(C, D)

 RUN v6b_mixed_s0 | mode mixed | 60 phases after phase 0 | device cuda
 phase order: init (random) → mix → mix → mix → mix → mix → mix → mix → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 60,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)
phase 0: fresh random model (66.8M params)

 PHASE 0 DONE | trained: init | starting model, no training
--------------------------------------------------------------------------------------
   lang      val       Δ |   chrF      Δ |  chrF (unrestricted)  val (unrestr.)
   hi      8.965         |    1.2        |                  0.4          10.510
   zh      9.641         |    0.0        |                  0.0          10.526
   ar      9.196         |    2.7        |                  1.1          10.542

▶ PHASE 1/60 | training mix (2,746,642 pairs, batches alterna

p000_init.pt:   0%|          | 0.00/267M [00:00<?, ?B/s]

  [p01 mix] step    500 | lr 2.0e-04 | train 7.373 | val 6.196 (best 6.196) | window Δ n/a | stable | 66k tok/s | phase 1m41s | run ETA: after first phase | health {'wnorm': 142.798, 'rank_enc1': 5.3, 'rank_dec1': 9.3, 'rarely_active': 0.0201}
  [p01 mix] step  1,000 | lr 2.0e-04 | train 6.342 | val 5.501 (best 5.501) | window Δ n/a | stable | 70k tok/s | phase 3m15s | run ETA: after first phase | health {'wnorm': 146.268, 'rank_enc1': 11.1, 'rank_dec1': 32.2, 'rarely_active': 0.008}
  [p01 mix] step  1,500 | lr 2.0e-04 | train 5.878 | val 5.051 (best 5.051) | window Δ n/a | stable | 70k tok/s | phase 4m50s | run ETA: after first phase | health {'wnorm': 150.097, 'rank_enc1': 16.8, 'rank_dec1': 66.9, 'rarely_active': 0.0049}
  [p01 mix] step  2,000 | lr 2.0e-04 | train 5.534 | val 4.690 (best 4.690) | window Δ n/a | stable | 69k tok/s | phase 6m24s | run ETA: after first phase | health {'wnorm': 153.848, 'rank_enc1': 23.2, 'rank_dec1': 102.1, 'rarely_active': 0.0046}
  [p01 mix] step  

p001_mix.pt:   0%|          | 0.00/267M [00:00<?, ?B/s]

  [p02 mix] step    500 | lr 2.0e-04 | train 2.924 | val 2.004 (best 2.004) | window Δ n/a | stable | 66k tok/s | phase 1m40s | run ETA ~125h14m | health {'wnorm': 231.232, 'rank_enc1': 640.0, 'rank_dec1': 504.5, 'rarely_active': 0.0206}
  [p02 mix] step  1,000 | lr 2.0e-04 | train 3.023 | val 2.025 (best 2.004) | window Δ n/a | stable | 70k tok/s | phase 3m14s | run ETA ~125h14m | health {'wnorm': 231.695, 'rank_enc1': 631.2, 'rank_dec1': 492.5, 'rarely_active': 0.0211}
  [p02 mix] step  1,500 | lr 2.0e-04 | train 3.054 | val 2.024 (best 2.004) | window Δ n/a | stable | 70k tok/s | phase 4m48s | run ETA ~125h14m | health {'wnorm': 232.071, 'rank_enc1': 659.2, 'rank_dec1': 485.4, 'rarely_active': 0.0202}
  [p02 mix] step  2,000 | lr 2.0e-04 | train 3.055 | val 2.031 (best 2.004) | window Δ n/a | stable | 70k tok/s | phase 6m22s | run ETA ~125h14m | health {'wnorm': 232.422, 'rank_enc1': 646.7, 'rank_dec1': 510.7, 'rarely_active': 0.0207}
  [p02 mix] step  2,500 | lr 2.0e-04 | train 3.0

...

No files have been modified since last commit. Skipping to prevent empty commit.


  [p38 mix] step  1,000 | lr 2.0e-04 | train 2.796 | val 1.873 (best 1.845) | window Δ n/a | stable | 70k tok/s | phase 3m13s | run ETA ~8h30m | health {'wnorm': 231.186, 'rank_enc1': 956.7, 'rank_dec1': 593.1, 'rarely_active': 0.055}
  [p38 mix] step  1,500 | lr 2.0e-04 | train 2.808 | val 1.871 (best 1.845) | window Δ n/a | stable | 70k tok/s | phase 4m47s | run ETA ~8h30m | health {'wnorm': 231.484, 'rank_enc1': 965.3, 'rank_dec1': 603.5, 'rarely_active': 0.0527}
  [p38 mix] step  2,000 | lr 2.0e-04 | train 2.836 | val 1.871 (best 1.845) | window Δ n/a | stable | 70k tok/s | phase 6m22s | run ETA ~8h30m | health {'wnorm': 231.753, 'rank_enc1': 970.1, 'rank_dec1': 606.6, 'rarely_active': 0.0532}
  [p38 mix] step  2,500 | lr 2.0e-04 | train 2.856 | val 1.871 (best 1.845) | window Δ n/a | stable | 70k tok/s | phase 7m56s | run ETA ~8h30m | health {'wnorm': 232.005, 'rank_enc1': 957.2, 'rank_dec1': 599.9, 'rarely_active': 0.0541}
  [p38 mix] step  3,000 | lr 2.0e-04 | train 2.860 | val 

p038_mix.pt:   0%|          | 0.00/267M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


  [p39 mix] step    500 | lr 2.0e-04 | train 2.688 | val 1.850 (best 1.850) | window Δ n/a | stable | 66k tok/s | phase 1m40s | run ETA ~8h06m | health {'wnorm': 230.953, 'rank_enc1': 974.4, 'rank_dec1': 594.0, 'rarely_active': 0.0541}
  [p39 mix] step  1,000 | lr 2.0e-04 | train 2.794 | val 1.877 (best 1.850) | window Δ n/a | stable | 70k tok/s | phase 3m14s | run ETA ~8h06m | health {'wnorm': 231.28, 'rank_enc1': 959.3, 'rank_dec1': 591.9, 'rarely_active': 0.0592}


In [ ]:
print_run_summary(RUN)